# TimesFM — Recursive Block Forecasting (Ablation)

Follow-up experiment to `model_experiment_TimesFM.ipynb`, isolating one
specific question that notebook's own Plot 1 raised: is the bad
52-week-direct holdout WMAE (**2618.40**, vs. ~1716-1810 at the 13-week
CV horizon) caused by the *length* of the forecast horizon, or is it a
fundamental limit of zero-shot forecasting on this data?

**Same setup, one variable changed.** Same data, same local train/test
split, same model checkpoint, same `context_length=52` (the winning
value from the original notebook's Step 5 sweep, reused here rather than
re-swept — this notebook isn't re-tuning context length, it's testing a
different *forecasting strategy* for the horizon). The only thing that
changes is **how the 52-week horizon gets forecast**:

- Original notebook: one direct `forecast_series(..., horizon=52)` call —
  the model has to commit to all 52 future weeks at once, with zero
  calendar signal (this project's TimesFM notebooks deliberately use no
  covariates) to tell it which of those weeks is Thanksgiving.
- This notebook: `recursive_forecast_series` — forecast 13 weeks at a
  time (`BLOCK_SIZE = HORIZON_CV = 13`, the same horizon that scored well
  during CV), feeding each block's own forecast back as additional
  context before forecasting the next block. Still zero calendar signal
  — this doesn't fix the "doesn't know when Thanksgiving is" problem —
  but each individual block only has to commit to a much shorter, less
  phase-uncertain horizon.

If recursive blocking meaningfully closes the gap, that points to
*horizon length* as the main driver. If it doesn't, that points more to
the missing calendar signal — the harder problem, and the one
`forecast_with_covariates()` (not attempted in this project — needs the
`timesfm[xreg]` extra) would actually address.

**Not executed locally against the full dataset** — designed and unit-
tested (`recursive_forecast_series` in `utils/timesfm_model.py`) against
the real pretrained checkpoint on synthetic data: confirmed a single-block
call matches `forecast_series` exactly, multi-block shapes/no-NaN are
correct, and recursive genuinely diverges from a direct call after the
first block. Meant to be run on a GPU runtime (Colab) — full-scale
recursive evaluation costs ~4x a direct call (4 sequential forward passes
instead of 1), which is fine on GPU but slow on CPU.

<a id='1'></a>
## 1. Setup

In [ ]:
!git clone https://github.com/tamari1990/ml-final-project.git
%cd ml-final-project
!git pull

In [ ]:
!pip install -q timesfm mlflow dagshub

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils.timesfm_model import (
    load_timesfm, forecast_series, recursive_forecast_series, min_context_length,
    build_full_calendar_panel, series_arrays_from_panel, CHECKPOINT,
)
from utils.metrics import wmae

pd.set_option('display.max_columns', 50)

SEED = 42
np.random.seed(SEED)

DATA_DIR = 'data/raw/walmart-recruiting-store-sales-forecasting/'

train = pd.read_csv(DATA_DIR + 'train.csv', parse_dates=['Date'])
train = train.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

print(f'train : {train.shape}, {train.Date.min().date()} -> {train.Date.max().date()}, '
      f'{train.Date.nunique()} weeks, {train[["Store","Dept"]].drop_duplicates().shape[0]} series')

In [ ]:
MAX_CONTEXT = 52
MAX_HORIZON = 52

print('Downloading/loading google/timesfm-2.5-200m-pytorch (~200M parameters, first run only)...')
tfm = load_timesfm(max_context=MAX_CONTEXT, max_horizon=MAX_HORIZON)
print('TimesFM loaded and compiled.')
print('device:', tfm.model.device)

<a id='2'></a>
## 2. Local Train/Test Split

Identical split to `model_experiment_TimesFM.ipynb` (same reasoning: last
52 weeks of `train.csv` held out locally).

In [ ]:
unique_dates = np.sort(train['Date'].unique())
cutoff_date = unique_dates[-52]

local_train_raw = train[train['Date'] < cutoff_date].copy()
local_test_raw = train[train['Date'] >= cutoff_date].copy()
local_train_dates = np.sort(local_train_raw['Date'].unique())

print(f'cutoff date: {pd.Timestamp(cutoff_date).date()}')
print(f'local_train_raw: {local_train_raw.shape}, {local_train_raw.Date.nunique()} weeks')
print(f'local_test_raw : {local_test_raw.shape}, {local_test_raw.Date.nunique()} weeks')

<a id='3'></a>
## 3. Recursive Holdout Evaluation

`CONTEXT_LENGTH = 52` (reused, the winning value from the original
notebook's Step 5 sweep) and `BLOCK_SIZE = 13` (`= HORIZON_CV`, the
horizon that scored well during CV). `HORIZON_FINAL = 52 = 4 blocks`
exactly (`52 / 13 = 4`).

Scored against the **full** local-test holdout set (same ~3,200-series
scale as the original notebook's Step 7), not a subsample — this is the
one "official" number being compared directly against `2618.40`.

In [ ]:
CONTEXT_LENGTH = 52
BLOCK_SIZE = 13
HORIZON_FINAL = 52
DIRECT_HOLDOUT_WMAE = 2618.40  # from model_experiment_TimesFM.ipynb's Step 7, reused for comparison

assert CONTEXT_LENGTH >= min_context_length(tfm)
assert HORIZON_FINAL % BLOCK_SIZE == 0
print(f'context_length={CONTEXT_LENGTH}, block_size={BLOCK_SIZE}, '
      f'n_blocks={HORIZON_FINAL // BLOCK_SIZE}, comparing against direct holdout WMAE={DIRECT_HOLDOUT_WMAE}')

In [ ]:
holdout_panel = build_full_calendar_panel(pd.concat([local_train_raw, local_test_raw], ignore_index=True))
holdout_arrays = series_arrays_from_panel(holdout_panel)

local_train_panel = build_full_calendar_panel(local_train_raw)
local_train_arrays = series_arrays_from_panel(local_train_panel)
n_train_weeks = len(local_train_dates)

keys = [k for k in local_train_arrays if k in holdout_arrays]
contexts = [local_train_arrays[k][0][-CONTEXT_LENGTH:] for k in keys]
print(f'{len(keys)} series in the holdout set')

In [ ]:
recursive_point = recursive_forecast_series(tfm, contexts, total_horizon=HORIZON_FINAL, block_size=BLOCK_SIZE)
recursive_point = np.clip(recursive_point, 0, None)

recursive_pred_rows = []
for i, key in enumerate(keys):
    sales, holiday, dates_arr = holdout_arrays[key]
    test_dates = dates_arr[n_train_weeks: n_train_weeks + HORIZON_FINAL]
    test_actual = sales[n_train_weeks: n_train_weeks + HORIZON_FINAL]
    test_holiday = holiday[n_train_weeks: n_train_weeks + HORIZON_FINAL].astype(bool)
    for step, (d, a, p, h) in enumerate(zip(test_dates, test_actual, recursive_point[i], test_holiday)):
        recursive_pred_rows.append((key[0], key[1], pd.Timestamp(d), a, p, h))

recursive_pred_df = pd.DataFrame(recursive_pred_rows, columns=['Store', 'Dept', 'Date', 'Actual', 'Predicted', 'IsHoliday'])
recursive_pred_df['Residual'] = recursive_pred_df['Actual'] - recursive_pred_df['Predicted']

recursive_holdout_wmae = wmae(recursive_pred_df['Actual'], recursive_pred_df['Predicted'], recursive_pred_df['IsHoliday'])
print(f'Recursive holdout WMAE (context_length={CONTEXT_LENGTH}, block_size={BLOCK_SIZE}): {recursive_holdout_wmae:.2f}')
print(f'Direct holdout WMAE (from original notebook)                 : {DIRECT_HOLDOUT_WMAE:.2f}')
improvement_pct = 100 * (DIRECT_HOLDOUT_WMAE - recursive_holdout_wmae) / DIRECT_HOLDOUT_WMAE
print(f'{"Improvement" if improvement_pct > 0 else "Regression"}: {abs(improvement_pct):.1f}%')

<a id='4'></a>
## 4. Plots

**Plot 1 — Actual vs. direct vs. recursive**, the same 3 sample
Store/Dept combos as the original notebook's Plot 1 (`(1, 1)`, `(1, 72)`,
`(20, 1)`) — the direct-52-week line is recomputed here (cheap, only 3
series) rather than hardcoded, so it's pulled from this same run's model
and data, not a screenshot.

In [ ]:
sample_combos = [(1, 1), (1, 72), (20, 1)]
sample_contexts = [local_train_arrays[k][0][-CONTEXT_LENGTH:] for k in sample_combos]

sample_direct_point, _ = forecast_series(tfm, sample_contexts, horizon=HORIZON_FINAL)
sample_direct_point = np.clip(sample_direct_point, 0, None)

sample_recursive_point = recursive_forecast_series(tfm, sample_contexts, total_horizon=HORIZON_FINAL, block_size=BLOCK_SIZE)
sample_recursive_point = np.clip(sample_recursive_point, 0, None)

In [ ]:
fig, axes = plt.subplots(len(sample_combos), 1, figsize=(11, 10), sharex=True)
for ax, (store, dept), direct_p, recur_p in zip(axes, sample_combos, sample_direct_point, sample_recursive_point):
    sales, holiday, dates_arr = holdout_arrays[(store, dept)]
    test_dates = dates_arr[n_train_weeks: n_train_weeks + HORIZON_FINAL]
    test_actual = sales[n_train_weeks: n_train_weeks + HORIZON_FINAL]
    test_holiday = holiday[n_train_weeks: n_train_weeks + HORIZON_FINAL].astype(bool)

    ax.plot(test_dates, test_actual, label='Actual', marker='o', markersize=3, color='black')
    ax.plot(test_dates, direct_p, label='Direct (52-week)', marker='x', markersize=3, color='#C44E52')
    ax.plot(test_dates, recur_p, label='Recursive (4x13-week)', marker='s', markersize=3, color='#4C72B0')
    for hd in test_dates[test_holiday]:
        ax.axvline(hd, color='gray', alpha=0.3, linestyle='--')
    ax.set_title(f'Store {store}, Dept {dept}')
    ax.legend()
axes[-1].set_xlabel('Date')
fig.suptitle('TimesFM: actual vs. direct vs. recursive forecast — local-test holdout')
plt.tight_layout()
plt.savefig('plots/actual_vs_direct_vs_recursive_timesfm.png', dpi=150, bbox_inches='tight')
plt.show()

**Plot 2 — WMAE comparison**: direct vs. recursive, full holdout set.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
stages = ['Direct\n(52-week)', 'Recursive\n(4x13-week)']
values = [DIRECT_HOLDOUT_WMAE, recursive_holdout_wmae]
colors = ['#C44E52', '#4C72B0']
ax.bar(stages, values, color=colors)
for i, v in enumerate(values):
    ax.text(i, v + max(values) * 0.01, f'{v:.0f}', ha='center')
ax.set_ylabel('Holdout WMAE')
ax.set_title('TimesFM holdout WMAE: direct vs. recursive forecasting')
plt.tight_layout()
plt.savefig('plots/direct_vs_recursive_wmae_timesfm.png', dpi=150)
plt.show()

<a id='5'></a>
## 5. MLflow Logging (DagsHub-hosted)

Logged under the same `TimesFM_Training` experiment as the original
notebook, as a distinctly-named run — so both show up side by side in
the DagsHub UI for direct comparison.

In [ ]:
import dagshub

dagshub.init(repo_owner='tgela23', repo_name='ml-final-project', mlflow=True)

import mlflow
mlflow.set_experiment('TimesFM_Training')

with mlflow.start_run(run_name='TimesFM_Recursive_Holdout'):
    mlflow.log_param('checkpoint', CHECKPOINT)
    mlflow.log_param('context_length', CONTEXT_LENGTH)
    mlflow.log_param('block_size', BLOCK_SIZE)
    mlflow.log_param('horizon_final', HORIZON_FINAL)
    mlflow.log_param('n_blocks', HORIZON_FINAL // BLOCK_SIZE)

    mlflow.log_metric('direct_holdout_wmae', DIRECT_HOLDOUT_WMAE)
    mlflow.log_metric('recursive_holdout_wmae', recursive_holdout_wmae)
    mlflow.log_metric('improvement_pct', improvement_pct)

print(f'TimesFM_Recursive_Holdout run logged. recursive={recursive_holdout_wmae:.2f}, '
      f'direct={DIRECT_HOLDOUT_WMAE:.2f}, {"improvement" if improvement_pct > 0 else "regression"}={abs(improvement_pct):.1f}%')

## 6. Conclusion

If `recursive_holdout_wmae` meaningfully beats `2618.40`, that's evidence
the 52-week *horizon length* itself (not just the missing calendar
signal) was a major driver of the flat, spike-missing predictions in the
original notebook — and `TimesFMForecastPipeline` could reasonably be
updated to use `recursive_forecast_series` internally instead of one
direct call, the same way `n_blocks` is already computed in `predict()`
(currently used for one direct multi-block-length call, not a truly
recursive one).

If it doesn't meaningfully improve, that points more squarely at the
missing calendar signal as the real limitation — in which case
`forecast_with_covariates()` (not attempted in this project) would be the
more principled next thing to try, at the cost of the `timesfm[xreg]`
extra and its `jax` dependency.